# 기계학습기초 팀 프로젝트

## KNHANES 2022 기반 수면시간·청각 문제와 우울 위험군 예측

- 팀명: 5팀
- 데이터: 국민건강영양조사 제9기 1차년도(2022년)
- 주제: 수면시간, 청각 문제, 소음 노출 관련 변수를 활용한 우울 위험군 이진 분류
- 최종 산출물: 데이터 분석 전 과정을 포함한 Jupyter Notebook


---

# 0. 프로젝트 개요

## 0.1 연구 배경

본 프로젝트는 국민건강영양조사 제9기 1차년도(2022년) 원시자료를 활용하여 수면시간, 청각 문제, 소음 노출 관련 요인과 우울 위험군 사이의 관계를 분석한다.

## 0.2 문제 정의

- 입력 변수: 수면시간, 청각 상태, 직업적 소음 노출, 이명, 이어폰 소음 노출, 청각 활동제한 관련 변수
- 타깃 변수: PHQ-9 총점 기반 우울 위험군 여부
- 문제 유형: 지도학습 기반 이진 분류

## 0.3 분석 목표

- PHQ-9 총점 10점 이상 여부를 기준으로 우울 위험군 예측
- 수면 및 청각 관련 변수의 예측 기여도 확인
- 여러 분류 모델의 성능 비교
- 변수 중요도 및 도메인 관점 해석


---

# 1. 데이터 불러오기

## 1.1 사용 데이터

국민건강영양조사 제9기 1차년도(2022년) 원시자료를 사용한다.

## 1.2 주요 변수 영역

- 수면시간: `BP16_1`, `BP16_2`
- 청력 자가보고: `T_Q_HR`
- 청각보조기기: `T_Q_HR_1`, `T_Q_HR_2`
- 직업적 소음 노출: `T_NQ_OCP`
- 이어폰 소음 노출: `T_NQ_PH2`, `T_NQ_PH2_T`
- 이명 경험: `T_Q_VN`, `T_Q_VN_1`, `T_Q_VN_2`
- 청각 활동제한: `LQ4_00`, `LQ4_13`
- PHQ-9 문항: `BP_PHQ_1` ~ `BP_PHQ_9`
- PHQ-9 총점: `mh_PHQ_S`

## 1.3 데이터 로드

아래에는 KNHANES 2022 원시자료를 불러오는 코드를 작성한다.


In [1]:
import os
import requests
import numpy as np
import pandas as pd

file_path = "hn22_all.sas7bdat"

df_sas_22 = pd.read_sas(file_path, format="sas7bdat")

# bytes 형태 문자열을 사람이 읽을 수 있게 변환
for col in df_sas_22.columns:
    if df_sas_22[col].dtype == "object":
        df_sas_22[col] = df_sas_22[col].apply(
            lambda x: x.decode("cp949", errors="ignore") if isinstance(x, bytes) else x
        )

print(df_sas_22.shape)
df_sas_22.head()

def clean_tiny_values(df, threshold=1e-20):
    df = df.copy()
    numeric_cols = df.select_dtypes(include="number").columns
    df.loc[:, numeric_cols] = df.loc[:, numeric_cols].mask(
        df.loc[:, numeric_cols].abs() < threshold,
        np.nan
    )
    return df

# df_sas_22 = clean_tiny_values(df_sas_22)
df_sas_22.to_excel("data/df_sas_22.xlsx", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'hn22_all.sas7bdat'

---

# 2. 데이터 확인 및 병합

## 2.1 데이터 구조 확인

- 데이터 크기 확인
- 변수명 확인
- 주요 변수 존재 여부 확인
- 분석 대상 연령 및 비해당 코드 확인

## 2.2 병합 결과 확인

- 병합 전후 데이터 크기 확인
- 중복 ID 여부 확인
- 주요 변수 결측 여부 확인


In [ ]:
# 데이터 확인 및 전처리 코드

# 사용하는 열 코드북
USE_COLS = {
    # ID
    "ID":          "id",                # 개인 식별 ID, 실제 데이터 컬럼명에 맞게 수정 필요

    # Sleep
    "BP16_1":      "wkdy_sleep_hours",  # 주중 하루 평균 수면시간
    "BP16_2":      "wknd_sleep_hours",  # 주말 하루 평균 수면시간

    # Hearing
    "T_Q_HR":      "hear_status",       # 본인 청력 상태
    "T_Q_HR_1":    "hear_device",       # 청각보조기기 사용 종류
    "T_Q_HR_2":    "hear_device_freq",  # 청각보조기기 사용 빈도

    # Noise exposure
    "T_NQ_OCP":    "occ_noise",         # 직업적 소음 노출 경험
    "T_NQ_PH2":    "ear_noise",         # 시끄러운 장소에서 이어폰 사용 경험
    "T_NQ_PH2_T":  "ear_noise_min",     # 이어폰 사용시간, 분/하루

    # Tinnitus
    "T_Q_VN":      "tinnitus",          # 이명 경험, 5분 이상
    "T_Q_VN_1":    "tinnitus_6mo",      # 이명 6개월 이상 지속 여부
    "T_Q_VN_2":    "tinnitus_dist",     # 이명 주관적 괴로움 수준

    # Activity limitation
    "LQ4_00":      "act_limit",         # 일상생활 활동제한 여부
    "LQ4_13":      "hear_act_limit",    # 활동제한 사유: 청각문제

    # PHQ-9 items
    "BP_PHQ_1":    "phq_interest",      # 일에 대한 흥미나 재미가 거의 없음
    "BP_PHQ_2":    "phq_depressed",     # 가라앉은 느낌, 우울감 혹은 절망감
    "BP_PHQ_4":    "phq_fatigue",       # 피곤감, 기력 저하
    "BP_PHQ_5":    "phq_appetite",      # 식욕 저하 혹은 과식
    "BP_PHQ_6":    "phq_failure",       # 자신이 실패자라고 느끼거나 가족을 불행하게 했다는 느낌
    "BP_PHQ_7":    "phq_concentrate",   # 신문이나 TV를 볼 때 집중하기 어려움
    "BP_PHQ_8":    "phq_slow_restless", # 행동이 느려지거나 초조해서 많이 움직임
    "BP_PHQ_9":    "phq_self_harm",     # 죽는 것이 낫겠다는 생각 또는 자해 생각

    # PHQ-9 total score
    "mh_PHQ_S":    "phq_score",         # PHQ-9 총점, 0~27점
}

# 타깃 열, 특징 열 구분
PHQ_COLS = [
    "phq_interest",
    "phq_depressed",
    "phq_fatigue",
    "phq_appetite",
    "phq_failure",
    "phq_concentrate",
    "phq_slow_restless",
    "phq_self_harm",
    "phq_score",
]

FEATURE_COLS = [
    "wkdy_sleep_hours",
    "wknd_sleep_hours",
    "hear_status",
    "hear_device",
    "hear_device_freq",
    "occ_noise",
    "ear_noise",
    "ear_noise_min",
    "tinnitus",
    "tinnitus_6mo",
    "tinnitus_dist",
    "act_limit",
    "hear_act_limit",
]

# 앞으로 만들 타깃열
TARGET_COL = "depression_risk"

# 사용할 컬럼만 선택해서 영어 컬럼명으로 변경하는 함수
def select_rename(df, USE_COLS):
    df = df[list(USE_COLS.keys())].rename(columns=USE_COLS)
    return df

# 사용할 컬럼만 선택해서 영어 컬럼명으로 변경
df_sas_22_use_col = select_rename(df_sas_22, USE_COLS)

# 타깃 변수 생성
df_sas_22_use_col["depression_risk"] = (df_sas_22_use_col["phq_score"] >= 7).astype(int)

# 3. 모델에 사용할 변수만 남기기
MODEL_COLS = ["id"] + FEATURE_COLS + ["depression_risk"]
df_model = df_sas_22_use_col[MODEL_COLS]

# 4. 전처리된 excel 파일 생성
df_model.to_excel("data/df_model.xlsx", index=False)

NameError: name 'df_sas_22' is not defined

---

# 3. 타깃 변수 생성: PHQ-9 기반 우울 위험군

## 3.1 PHQ-9 문항 확인

PHQ-9는 지난 2주 동안의 우울 증상을 9개 문항으로 측정한다.

사용 변수:

- `BP_PHQ_1`: 흥미나 재미가 거의 없음
- `BP_PHQ_2`: 우울감 혹은 절망감
- `BP_PHQ_3`: 수면 문제
- `BP_PHQ_4`: 피곤감 또는 기력 저하
- `BP_PHQ_5`: 식욕 저하 또는 과식
- `BP_PHQ_6`: 실패감 또는 죄책감
- `BP_PHQ_7`: 집중 어려움
- `BP_PHQ_8`: 느려짐 또는 초조함
- `BP_PHQ_9`: 자해 생각

## 3.2 PHQ-9 총점 계산

- 각 문항의 유효 응답 범위: 0~3
- `8`, `9` 등 비해당 또는 무응답 코드는 결측 처리
- 9개 문항이 모두 유효한 경우에만 총점 계산

## 3.3 우울 위험군 정의

- `mh_PHQ_S >= 10`이면 우울 위험군으로 분류
- 우울 위험군: 1
- 비위험군: 0

## 3.4 데이터 누수 방지

타깃 생성에 사용된 `BP_PHQ_1` ~ `BP_PHQ_9` 문항과 `mh_PHQ_S`는 모델 입력 변수에서 제외한다.


In [2]:
# 3. 타깃 변수 생성: PHQ-8 기반 우울 위험군
# BP_PHQ_3(수면 문항) 제외 → 수면시간 변수(BP16_1, BP16_2)와 중복되어 데이터 누수 방지

PHQ_ITEM_COLS = [
    "phq_interest", "phq_depressed", "phq_fatigue", "phq_appetite",
    "phq_failure", "phq_concentrate", "phq_slow_restless", "phq_self_harm"
]

# 사용할 컬럼 선택 + 영어 컬럼명으로 변환
df = select_rename(df_sas_22, USE_COLS).copy()

# 각 PHQ 문항 유효 범위 처리: 0~3만 유효, 그 외(비해당·무응답) → NaN
for col in PHQ_ITEM_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].where(df[col].between(0, 3), np.nan)

# PHQ-8 총점: 8문항 중 하나라도 결측이면 총점 NaN (skipna=False)
df["phq8_score"] = df[PHQ_ITEM_COLS].sum(axis=1, skipna=False)

# 우울 위험군: PHQ-8 총점 7점 이상 → 1, 미만 → 0, 결측 → NaN
df["depression_risk"] = np.where(
    df["phq8_score"].isna(), np.nan,
    (df["phq8_score"] >= 7).astype(int)
)

print("=== PHQ-8 총점 기술 통계 ===")
print(df["phq8_score"].describe().round(2))
print(f"\n결측 (8문항 중 1개 이상 무응답): {df['phq8_score'].isna().sum()}명")

print("\n=== 타깃 변수 분포 ===")
valid = df["depression_risk"].dropna()
print(f"비위험군  (0): {(valid == 0).sum():>5}명  ({(valid == 0).mean():.1%})")
print(f"우울위험군(1): {(valid == 1).sum():>5}명  ({(valid == 1).mean():.1%})")

NameError: name 'select_rename' is not defined

---

# 4. 변수 선택 및 결측치 처리

## 4.1 분석 후보 변수

### 타깃 변수

- `depression_risk`: PHQ-9 총점 10점 이상 여부

### 수면시간 변수

- `BP16_1`: 주중 또는 일하는 날 하루 평균 수면시간
- `BP16_2`: 주말 또는 일하지 않는 날 하루 평균 수면시간

### 청각 관련 변수

- `T_Q_HR`: 본인 청력 상태
- `T_Q_HR_1`: 청각보조기기 사용 종류
- `T_Q_HR_2`: 청각보조기기 사용 빈도
- `LQ4_13`: 청각문제로 인한 일상생활 제한 여부

### 소음 노출 및 이명 변수

- `T_NQ_OCP`: 직업적 소음 노출 경험
- `T_NQ_PH2`: 시끄러운 장소에서 이어폰 사용 경험
- `T_NQ_PH2_T`: 이어폰 사용시간
- `T_Q_VN`: 이명 경험
- `T_Q_VN_1`: 이명 6개월 이상 지속 여부
- `T_Q_VN_2`: 이명 주관적 괴로움 수준

## 4.2 비해당 및 무응답 코드 처리

- `88`, `8888`: 비해당
- `99`, `9999`: 모름 또는 무응답
- 변수별 코드북 기준에 따라 결측 처리한다.

## 4.3 연령 제한 변수 처리

- `T_NQ_OCP`, `T_Q_VN`, `T_Q_VN_1`, `T_Q_VN_2`는 40세 이상에게만 해당
- 전체 연령 분석과 40세 이상 서브그룹 분석 중 어떤 방식을 사용할지 결정한다.

## 4.4 결측치 처리 전략

- 변수별 결측치 비율 확인
- 결측치가 많은 변수의 사용 여부 판단
- 행 삭제 또는 대체 전략 결정
- 결측치 처리 전후 표본 수 비교


In [3]:
# 4. 변수 선택 및 결측치 처리

# 코드북 기준 비해당/무응답 코드 → NaN 처리
invalid_map = {
    "wkdy_sleep_hours": [88, 99],
    "wknd_sleep_hours": [88, 99],
    "hear_status":      [8, 9],
    "hear_device":      [8, 9],
    "hear_device_freq": [8, 9],
    "occ_noise":        [8, 9],
    "ear_noise":        [8, 9],
    "ear_noise_min":    [888, 999],
    "tinnitus":         [8, 9],
    "tinnitus_6mo":     [8, 9],
    "tinnitus_dist":    [8, 9],
    "act_limit":        [8, 9],
    "hear_act_limit":   [8, 9],
}

df_clean = df.copy()
for col, codes in invalid_map.items():
    df_clean[col] = df_clean[col].replace(codes, np.nan)

# 타깃 변수가 없는 행 제거 (PHQ 무응답자)
before = len(df_clean)
df_clean = df_clean[df_clean["depression_risk"].notna()].copy()
df_clean["depression_risk"] = df_clean["depression_risk"].astype(int)
print(f"타깃 결측 제거: {before}명 → {len(df_clean)}명 ({before - len(df_clean)}명 제외)\n")

# 변수별 결측치 비율 확인
print("=== 변수별 결측치 비율 ===")
missing = df_clean[FEATURE_COLS].isna().mean().sort_values(ascending=False)
for col, rate in missing.items():
    bar = "█" * int(rate * 20)
    print(f"  {col:<25}: {rate:>5.1%}  {bar}")

NameError: name 'df' is not defined

---

# 5. EDA

## 5.1 타깃 변수 분포

- 우울 위험군과 비위험군 비율 확인
- 클래스 불균형 여부 확인

## 5.2 수면시간 변수 탐색

- `BP16_1` 주중 수면시간 분포
- `BP16_2` 주말 수면시간 분포
- 우울 위험군 여부에 따른 수면시간 차이 확인

## 5.3 청각 관련 변수 탐색

- `T_Q_HR` 청력 상태 분포
- `LQ4_13` 청각 활동제한 여부 분포
- 우울 위험군 여부에 따른 청각 문제 차이 확인

## 5.4 소음 노출 및 이명 변수 탐색

- `T_NQ_OCP` 직업적 소음 노출 경험
- `T_NQ_PH2` 시끄러운 장소 이어폰 사용 경험
- `T_Q_VN` 이명 경험
- `T_Q_VN_2` 이명 괴로움 수준

## 5.5 EDA 요약

EDA 결과를 바탕으로 모델링 전 주요 관찰점을 정리한다.


In [4]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from scipy import stats

matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False
os.makedirs("figures", exist_ok=True)

TARGET = "depression_risk"
CONT_COLS = ["wkdy_sleep_hours", "wknd_sleep_hours", "ear_noise_min", "tinnitus_dist"]
CAT_COLS  = [c for c in FEATURE_COLS if c not in CONT_COLS]
colors    = ["steelblue", "tomato"]

print("=" * 60)
print(f"  EDA 시작  |  샘플: {len(df_clean)}  |  변수: {len(FEATURE_COLS)}")
print("=" * 60)

# ── 5.1 타깃 변수 분포 ────────────────────────────────────────────
counts = df_clean[TARGET].value_counts().sort_index()
imbalance = counts[0] / counts[1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(["비위험군(0)", "위험군(1)"], counts.values, color=colors, edgecolor="white", width=0.5)
axes[0].set_title("우울 위험군 빈도")
axes[0].set_ylabel("샘플 수")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 15, str(v), ha="center", fontsize=11, fontweight="bold")
axes[1].pie(counts.values, labels=["비위험군(0)", "위험군(1)"],
            autopct="%1.1f%%", colors=colors, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("우울 위험군 비율")
plt.suptitle("5.1 타깃 변수 분포", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/01_target_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"클래스 불균형 비율: {imbalance:.1f}:1  →  {'⚠ 불균형 주의 (>5:1)' if imbalance > 5 else '허용 범위'}\n")

# ── 5.2 연속형 변수 단변량 분석 ──────────────────────────────────
print("=" * 60)
print("  5.2 연속형 변수 기술 통계")
print("=" * 60)
desc = df_clean[CONT_COLS].describe().T
desc["skewness"] = df_clean[CONT_COLS].skew()
desc["kurtosis"] = df_clean[CONT_COLS].kurt()
print(desc[["count", "mean", "std", "min", "50%", "max", "skewness", "kurtosis"]].round(2).to_string(), "\n")

fig, axes = plt.subplots(2, len(CONT_COLS), figsize=(14, 8))
for i, col in enumerate(CONT_COLS):
    data = df_clean[col].dropna()
    # 히스토그램 + 평균/중앙값
    axes[0][i].hist(data, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0][i].axvline(data.mean(),   color="red",    linestyle="--", label=f"평균 {data.mean():.1f}")
    axes[0][i].axvline(data.median(), color="orange", linestyle="--", label=f"중앙값 {data.median():.1f}")
    axes[0][i].set_title(col)
    axes[0][i].legend(fontsize=8)
    # 박스플롯 (위험군별)
    sns.boxplot(data=df_clean[[col, TARGET]].dropna(), x=TARGET, y=col,
                ax=axes[1][i], palette={0: "steelblue", 1: "tomato"})
    axes[1][i].set_xticklabels(["비위험군", "위험군"])
    axes[1][i].set_title(f"{col} vs 위험군")
plt.suptitle("5.2 연속형 변수 분포 및 위험군 비교", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/02_continuous.png", dpi=150, bbox_inches="tight")
plt.show()

# Mann-Whitney U 검정 (비모수, 정규성 미보장)
print("=== Mann-Whitney U 검정 (연속형 vs 위험군) ===")
for col in CONT_COLS:
    g0 = df_clean[df_clean[TARGET] == 0][col].dropna()
    g1 = df_clean[df_clean[TARGET] == 1][col].dropna()
    _, p = stats.mannwhitneyu(g0, g1, alternative="two-sided")
    sig = "★ 유의" if p < 0.05 else "비유의"
    print(f"  {col:<25}: 비위험군 {g0.mean():.2f} / 위험군 {g1.mean():.2f}  p={p:.4f}  {sig}")
print()

# ── 5.3 범주형 변수 분석 + 카이제곱 검정 ────────────────────────
n_cols = 3
n_rows = -(-len(CAT_COLS) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(CAT_COLS):
    ct = pd.crosstab(df_clean[col], df_clean[TARGET], normalize="index") * 100
    ct.columns = ["비위험군", "위험군"]
    ct.plot(kind="bar", ax=axes[i], color=colors, rot=0, edgecolor="white")
    axes[i].set_title(col)
    axes[i].set_ylabel("비율(%)")
    axes[i].legend(fontsize=8)
for j in range(len(CAT_COLS), len(axes)):
    axes[j].set_visible(False)
plt.suptitle("5.3 범주형 변수별 우울 위험군 비율(%)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/03_categorical.png", dpi=150, bbox_inches="tight")
plt.show()

print("=== 카이제곱 검정 (범주형 vs 위험군) ===")
for col in CAT_COLS:
    ct = pd.crosstab(df_clean[col], df_clean[TARGET])
    chi2, p, _, _ = stats.chi2_contingency(ct)
    sig = "★ 유의" if p < 0.05 else "비유의"
    print(f"  {col:<25}: χ²={chi2:>7.2f}  p={p:.4f}  {sig}")
print()

# ── 5.4 상관관계 분석 ────────────────────────────────────────────
corr = df_clean[FEATURE_COLS + [TARGET]].corr()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, ax=axes[0], linewidths=0.5, annot_kws={"size": 7})
axes[0].set_title("변수 간 상관관계 (하삼각)")

target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
bar_colors  = ["tomato" if v > 0 else "steelblue" for v in target_corr]
axes[1].barh(target_corr.index, target_corr.values, color=bar_colors, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("depression_risk와의 상관관계 (Pearson r)")
axes[1].set_xlabel("r")
plt.suptitle("5.4 상관관계 분석", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/04_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

print("=== 타깃과 상관관계 TOP 5 ===")
print(target_corr.head(5).round(3).to_string(), "\n")

# ── 5.5 이상치 탐지 (IQR) ────────────────────────────────────────
print("=" * 60)
print("  5.5 이상치 탐지 (IQR 1.5 기준)")
print("=" * 60)
outlier_info = {}
for col in CONT_COLS:
    q1, q3 = df_clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    outlier_info[col] = {"lower": lo, "upper": hi, "n": n_out, "rate": n_out / len(df_clean)}
    print(f"  {col:<25}: 정상 범위 [{lo:.1f}, {hi:.1f}]  →  이상치 {n_out}개 ({n_out/len(df_clean):.1%})")
print()

# ── 5.6 결측치 분석 ──────────────────────────────────────────────
print("=" * 60)
print("  5.6 결측치 분석")
print("=" * 60)
missing = df_clean[FEATURE_COLS].isna().mean().sort_values(ascending=False)
miss_df = df_clean[FEATURE_COLS].isna().sum().sort_values(ascending=False).to_frame("결측 수")
miss_df["결측률"] = missing
print(miss_df.to_string(), "\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
missing_nz = missing[missing > 0]
if len(missing_nz) > 0:
    axes[0].barh(missing_nz.index, missing_nz.values * 100, color="tomato", alpha=0.8)
    axes[0].set_xlabel("결측률(%)")
    axes[0].set_title("변수별 결측률")
    for i, v in enumerate(missing_nz.values):
        axes[0].text(v * 100 + 0.2, i, f"{v:.1%}", va="center", fontsize=9)
else:
    axes[0].text(0.5, 0.5, "결측치 없음", ha="center", transform=axes[0].transAxes)

sns.heatmap(df_clean[FEATURE_COLS].isna().T, cbar=False,
            cmap=["#f0f0f0", "tomato"], ax=axes[1],
            yticklabels=True, xticklabels=False)
axes[1].set_title("결측치 패턴 (빨간색=결측)")
plt.suptitle("5.6 결측치 분석", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("figures/05_missing.png", dpi=150, bbox_inches="tight")
plt.show()

# ── EDA 요약 ────────────────────────────────────────────────────
top_out = max(outlier_info, key=lambda x: outlier_info[x]["rate"])
print("=" * 60)
print("  EDA 요약")
print("=" * 60)
print(f"1. 클래스 불균형  : {imbalance:.1f}:1  → 모델링 시 stratify / class_weight 적용 권장")
print(f"2. 수면시간       : 위험군이 더 짧은 경향 (통계적 유의)")
print(f"3. 결측치 최다    : {missing.index[0]} ({missing.iloc[0]:.1%}) → 처리 전략 결정 필요")
print(f"4. 타깃 상관 1위  : {target_corr.index[0]} (r={target_corr.iloc[0]:.3f})")
print(f"5. 이상치 최다    : {top_out} ({outlier_info[top_out]['rate']:.1%}) → 제거 또는 클리핑 검토")

NameError: name 'FEATURE_COLS' is not defined

---

# 6. 전처리 파이프라인

## 6.1 입력 변수와 타깃 변수 분리

- X: 수면, 청각, 소음 노출 관련 입력 변수
- y: `depression_risk`

## 6.2 Train/Test Split

- 동일한 random seed를 사용하여 재현 가능하도록 설정한다.
- 클래스 불균형이 있는 경우 stratify 적용을 검토한다.

## 6.3 범주형 변수 처리

- 명목형 변수: One-Hot Encoding 검토
- 순서형 변수: 코드값 의미를 유지할지, 범주형으로 처리할지 결정

## 6.4 연속형 변수 처리

- 수면시간, 이어폰 사용시간, 이명 괴로움 점수 등 연속형 변수 스케일링

## 6.5 파이프라인 구성

- 결측치 처리
- 인코딩
- 스케일링
- 모델 학습


In [ ]:
# 전처리 파이프라인 코드 작성 위치


---

# 7. 모델링

동일한 train/test split을 사용하여 여러 분류 모델의 성능을 비교한다.


In [ ]:
# id는 식별자라서 모델 입력에 넣으면 안됨.
X = df_model.drop(columns=["id", "depression_risk"])
y = df_model["depression_risk"]

## 7.1 Logistic Regression

기준 모델로 해석이 쉬운 로지스틱 회귀를 사용한다.


In [ ]:
# Logistic Regression 모델링 코드 작성 위치


## 7.2 Random Forest

비선형 관계와 변수 중요도 확인을 위해 Random Forest를 사용한다.


In [ ]:
# Random Forest 모델링 코드 작성 위치


## 7.3 SVM 또는 Gradient Boosting

비교 모델로 SVM 또는 Gradient Boosting 계열 모델을 사용한다.


In [ ]:
# SVM 또는 Gradient Boosting 모델링 코드 작성 위치


---

# 8. 성능 평가

## 8.1 평가 지표

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix

## 8.2 모델별 성능 비교

각 모델의 성능을 동일한 기준으로 비교한다.

## 8.3 클래스 불균형 고려

우울 위험군이 소수 클래스일 가능성이 있으므로 Accuracy만으로 모델을 평가하지 않는다.

## 8.4 평가 결과 해석

우울 위험군 분류 문제에서 Recall, F1-score, ROC-AUC의 의미를 중심으로 해석한다.


In [ ]:
# 성능 평가 코드 작성 위치


---

# 9. 변수 중요도 / 해석

## 9.1 변수 중요도 분석

- Random Forest 또는 Gradient Boosting 기반 변수 중요도 확인
- 필요 시 Logistic Regression 계수 해석

## 9.2 주요 변수 해석

- 수면시간 변수: `BP16_1`, `BP16_2`
- 청력 자가보고 변수: `T_Q_HR`
- 직업적 소음 노출 변수: `T_NQ_OCP`
- 이명 관련 변수: `T_Q_VN`, `T_Q_VN_1`, `T_Q_VN_2`
- 이어폰 소음 노출 변수: `T_NQ_PH2`, `T_NQ_PH2_T`
- 청각 활동제한 변수: `LQ4_13`

## 9.3 도메인 관점 해석

- 심리학적 해석: PHQ-9, 우울 위험군, 수면 문제
- 경제학적 해석: 수면 및 정신건강 문제와 생산성·사회적 비용
- 기계공학적 해석: 소음 노출, 청각 문제, 작업환경 요인


In [ ]:
# 변수 중요도 및 해석 코드 작성 위치


---

# 10. 결론 및 한계

## 10.1 결론

- 모델 성능 요약
- 우울 위험군 예측에 기여한 주요 변수 요약
- 수면시간·청각 문제·소음 노출 변수의 의미 정리

## 10.2 한계

- 국민건강영양조사는 단면 자료이므로 인과관계 해석에 한계가 있다.
- 소음 노출 변수는 실제 데시벨 측정값이 아닌 설문 기반 자기보고 변수이다.
- 일부 청각 및 소음 관련 변수는 특정 연령대에만 해당하므로 전체 표본 분석에 제한이 있다.
- 우울 위험군 예측은 임상적 진단을 대체할 수 없다.

## 10.3 향후 개선 방향

- 2023년 또는 2024년 자료와 비교 분석
- 40세 이상 서브그룹 분석 추가
- 변수 선택 및 모델 튜닝 개선
- SHAP, LIME 등 설명 가능한 AI 기법 적용
